# ノートブック4：しゃがみの補正・追加学習・3画面すべて正面の比較動画（ランタイム：A100・ハイメモリ）

ノートブック3の追加学習（`model_23998`）を出発点に、次を行います。

1. **お手本の 10〜12 秒の深いしゃがみを補正**（`edit_reference.py`）：脚の曲げを 4 割浅くして腰を上げ、腕のめり込みは肩を外へ開いて逃がす。区間の外は 1 フレームも変えない
2. **3,000 反復の追加学習**：ノートブック3の報酬設定を引き継ぎ、立ち位置の許容幅だけ 0.3 → **0.2 m** に締める（最大 24cm のドリフト対策）
3. **評価**：同じお手本の上で「ノートブック3の方策」と「今回の方策」を比べる。しゃがみ区間（9〜14 秒）は別に集計
4. **3画面すべて正面の比較動画**：左の人のメッシュは **ノートブック 04b（T4 で可）** で先に `2_global_front.mp4` を作っておく

**事前準備**：手元の `pai_final/scripts/` から次の 4 つをドライブの `pai_final/scripts/` に上書き（または追加）アップロードしておくこと。
`train_sim.py`（`--root-pos-std` を追加）／`edit_reference.py`（新規）／`gvhmr_to_g1_csv.py`（`--cam_azimuth` など）／`render_policy.py`（`--elevation`）

付録に、報酬の調整の効果を切り分ける比較実験（任意）を入れてあります。

## 1. ドライブの接続と設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
P = '/content/drive/MyDrive/pai_final'
os.environ['P'] = P

# 使うリポジトリのコミット（この版で動作を確認したソースに固定。切断後に入れ直しても同じ版になる）
PINS = {
    'GVHMR':            ('https://github.com/zju3dv/GVHMR',                  'ee960bb6e2ea2d381aa97f08e9b71ef320b624b1'),
    'GMR':              ('https://github.com/YanjieZe/GMR',                  'bb1bbe40774794fceb2a7c579a3464a28e68c844'),
    'unitree_rl_mjlab': ('https://github.com/unitreerobotics/unitree_rl_mjlab', '1425b15f73bd4095f0df53709d7c389c3eb9e790'),
    'mjlab':            ('https://github.com/mujocolab/mjlab',               '5af32e378dcb93c9e881ace83cc5a3f5d373fe60'),  # v1.2.0
}
for k, (url, rev) in PINS.items():
    os.environ[f'URL_{k}'], os.environ[f'REV_{k}'] = url, rev

# サビ後半 20 秒が、全編（伊原六花さんの動画 2分19秒）の何秒目か（音声の照合で特定済み）
SEG_START, SEG_END = 59.815, 80.78
os.environ['SEG_START'], os.environ['SEG_END'] = str(SEG_START), str(SEG_END)

# MuJoCo を画面なし（EGL）で描画するための設定
NVIDIA_ICD = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD):
    os.makedirs(os.path.dirname(NVIDIA_ICD), exist_ok=True)
    with open(NVIDIA_ICD, 'w') as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYTHONUNBUFFERED'] = '1'   # ログをすぐファイルに書き出す

# 動画・画像をノートブック上で表示する関数
from base64 import b64encode
from IPython.display import HTML, Image, display
def show(path, width=720):
    data = b64encode(open(path, 'rb').read()).decode()
    return HTML(f'<video width={width} controls loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

# ▼ ノートブック4の設定
BASE_REF = 'the_feels_full_g25'     # ノートブック3で作った、浮きを直したお手本
NAME = 'the_feels_full_g25s'        # 10〜12 秒のしゃがみを直したお手本（s = squat）
PREV_RUN = '2026-10-18_03-18-06_the_feels_full_g25_ft'   # 出発点：ノートブック3の追加学習
PREV_CKPT = 'model_23998.pt'
ITERS = 3000                        # 追加の反復回数（A100 で約 1.4 時間）
# しゃがみの補正
SQ_START, SQ_END, DEPTH_KEEP = 9.5, 13.5, 0.6   # 区間 [秒] と、脚の曲げを何割残すか（1.0 = 変更なし）
# 報酬：ノートブック3の設定を引き継ぎ、立ち位置の許容幅だけ締める（既定 0.3）
ACTION_RATE, BODY_POS_STD, BODY_ORI_STD, ROOT_POS_STD = -0.05, 0.25, 0.3, 0.2
# 正面カメラ（G1 の平均の向き yaw≈91° の反対側）
AZ, EL, DIST = 270, -5, 2.8
for k, v in dict(BASE_REF=BASE_REF, NAME=NAME, PREV_RUN=PREV_RUN, PREV_CKPT=PREV_CKPT, ITERS=ITERS,
                 SQ_START=SQ_START, SQ_END=SQ_END, DEPTH_KEEP=DEPTH_KEEP, ACTION_RATE=ACTION_RATE,
                 BODY_POS_STD=BODY_POS_STD, BODY_ORI_STD=BODY_ORI_STD, ROOT_POS_STD=ROOT_POS_STD,
                 AZ=AZ, EL=EL, DIST=DIST).items():
    os.environ[k] = str(v)
os.environ['TASK'] = 'Unitree-G1-Tracking'
os.environ['START'] = '0'
L = '/content/unitree_rl_mjlab'
os.environ['RUN_SPEEDTEST'] = '0'; os.environ['RESUME'] = '0'

!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls $P/outputs/motions | grep the_feels_full
!ls $P/logs/rsl_rl/g1_tracking/
!ls -la $P/outputs/gvhmr/the_feels_full/

## 2. unitree_rl_mjlab のインストール（5〜10分）

In [ ]:
%%bash
set -e
pip -q install uv
bash $P/scripts/pin_clone.sh $URL_unitree_rl_mjlab /content/unitree_rl_mjlab $REV_unitree_rl_mjlab
uv venv /content/venv_mjlab --python 3.11 -q --clear
cd /content/unitree_rl_mjlab
# mjlab 1.2.0（2026年3月公開）は MuJoCo 系の版を固定していないため、そのままだと最新版が入り
# 「mujoco と mujoco-warp の版が合わない」エラーになる。公開当時の組み合わせに固定する。
cat > /content/mjlab_pins.txt <<'EOF'
mujoco==3.5.0
mujoco-warp==3.5.0.2
warp-lang==1.12.0
EOF
uv pip install --python /content/venv_mjlab/bin/python -q --override /content/mjlab_pins.txt -e .     # mjlab==1.2.0 が入る
uv pip install --python /content/venv_mjlab/bin/python -q --override /content/mjlab_pins.txt matplotlib scipy   # scipy は mjlab の地形モジュールが使う（依存に書かれていない）
# 学習・変換・評価で使う部品をここでまとめて読み込んで確認（足りなければこのセルで止まる）
/content/venv_mjlab/bin/python - <<'EOF'
import sys, os; sys.path.insert(0, os.getcwd())
import torch, mjlab, mujoco, mujoco_warp, warp, scipy, importlib.metadata as m
import mjlab.entity, mjlab.scene, mjlab.envs, mjlab.rl, mjlab.tasks, src.tasks
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| mjlab', m.version('mjlab'),
      '| mujoco', mujoco.__version__, '| mujoco-warp', m.version('mujoco-warp'), '| warp', warp.__version__)
print('[ok] 必要な部品はすべて読み込めました')
EOF
# 学習ログはローカルに置き、ドライブへは監視セルが同期する（ドライブにあれば戻す）
mkdir -p logs/rsl_rl $P/logs/rsl_rl
bash $P/scripts/sync_logs.sh $P/logs/rsl_rl logs/rsl_rl
ls logs/rsl_rl/*/ 2>/dev/null | tail -5 || true

## 3. 10〜12 秒のしゃがみを補正したお手本を作る

`edit_reference.py` で補正 → 同じお手本を `gvhmr_to_g1_csv.py` に通して（接地補正・フィルタは掛けない）**正面からのプレビュー**を作ります。
出力の `pelvis_min_height_m`（腰の最低高さ）が上がり、`self_penetration_m` の `after_max` が 0.005 m 以下になっていれば成功です。
しゃがみをどこまで浅くするかは `DEPTH_KEEP`（1 の設定セル）で調整できます（0.6 = 曲げを 4 割浅く、0.75 = 2.5 割浅く）。

In [ ]:
%%bash
set -e
PY=/content/venv_mjlab/bin/python
for f in train_sim.py edit_reference.py gvhmr_to_g1_csv.py render_policy.py; do ls $P/scripts/$f; done
grep -q "root-pos-std" $P/scripts/train_sim.py || { echo "[stop] train_sim.py が古い版です"; exit 1; }
grep -q "cam_azimuth" $P/scripts/gvhmr_to_g1_csv.py || { echo "[stop] gvhmr_to_g1_csv.py が古い版です"; exit 1; }
grep -q -- "--elevation" $P/scripts/render_policy.py || { echo "[stop] render_policy.py が古い版です"; exit 1; }
uv pip install --python $PY -q imageio imageio-ffmpeg
bash $P/scripts/pin_clone.sh $URL_GMR /content/GMR $REV_GMR
cd /content
# (1) 補正（判定には mjlab の G1 を使う。腕の当たり判定があるため）
$PY $P/scripts/edit_reference.py --csv $P/outputs/motions/$BASE_REF.csv --out_csv $P/outputs/motions/$NAME.csv \
  --start $SQ_START --end $SQ_END --depth_keep $DEPTH_KEEP
# (2) 正面からのプレビュー（同じデータをそのまま通す：接地補正・推定ミス補間・フィルタはすべて無効）
MUJOCO_GL=egl $PY $P/scripts/gvhmr_to_g1_csv.py --from_csv $P/outputs/motions/$NAME.csv \
  --out_dir /content/front_ref --name $NAME --robot_xml /content/GMR/assets/unitree_g1/g1_mocap_29dof.xml \
  --ground none --no_outlier --filter none --cam_azimuth $AZ
cp /content/front_ref/${NAME}_retarget_preview.mp4 $P/outputs/motions/${NAME}_retarget_preview_front.mp4
ls -la $P/outputs/motions/ | grep $NAME

## 4. 物理チェック：しゃがみ区間（8〜15 秒）の新旧比較

**合格の目安**：補正後は腰の最低高さが上がり、自己干渉（めり込み）が 0.5cm 以下、横方向に必要な力（Fxy）が下がること。

In [ ]:
%%bash
set -e
PY=/content/venv_mjlab/bin/python
XML=$($PY -c "from mjlab.asset_zoo.robots.unitree_g1.g1_constants import G1_XML; print(G1_XML)")
$PY $P/scripts/check_reference.py --csv $P/outputs/motions/$NAME.csv --robot_xml $XML \
  --out_prefix $P/outputs/check/$NAME > $P/outputs/check/${NAME}_log.txt
$PY - <<EOF
import json, numpy as np
def load(n):
    c = np.genfromtxt(f"$P/outputs/check/{n}_check_frames.csv", delimiter=",", names=True)
    q = np.loadtxt(f"$P/outputs/motions/{n}.csv", delimiter=",")
    return c, q[:, 2]
(c0, z0), (c1, z1) = load("$BASE_REF"), load("$NAME")
t = c0["time_s"]
print(f"{'秒':>4} {'腰の最低[cm] 旧':>14} {'新':>5} {'めり込み[cm] 旧':>14} {'新':>5} {'横の力/mg 旧':>12} {'新':>5} {'滑り 旧':>7} {'新':>4}")
for s in range(8, 15):
    m = (t >= s) & (t < s + 1)
    print(f"{s:>4} {z0[m].min()*100:>14.1f} {z1[m].min()*100:>5.1f} {c0['self_pen_m'][m].max()*100:>14.1f} {c1['self_pen_m'][m].max()*100:>5.1f}"
          f" {c0['Fxy_per_mg'][m].max():>12.2f} {c1['Fxy_per_mg'][m].max():>5.2f} {int(c0['slip'][m].sum()):>7} {int(c1['slip'][m].sum()):>4}")
for n in ["$BASE_REF", "$NAME"]:
    j = json.load(open(f"$P/outputs/check/{n}_check.json"))
    print(n, j["frames_flagged"])
EOF

In [ ]:
# 正面からのプレビュー。9〜14 秒あたりで、しゃがみが不自然でないか・足が床を滑っていないかを見る
display(show(f"{P}/outputs/motions/{NAME}_retarget_preview_front.mp4", width=640))

## 5. 学習用の形式（npz）に変換

In [ ]:
%%bash
set -e
cd /content/unitree_rl_mjlab
for n in $BASE_REF $NAME; do
  cp $P/outputs/motions/$n.csv src/assets/motions/g1/
  if [ -f $P/outputs/motions/$n.npz ] && [ "$n" != "$NAME" ]; then cp $P/outputs/motions/$n.npz src/assets/motions/g1/; continue; fi
  /content/venv_mjlab/bin/python scripts/csv_to_npz.py --robot g1 --input-file src/assets/motions/g1/$n.csv \
    --output-name $n --input-fps 30 --output-fps 50
  cp src/assets/motions/g1/$n.npz $P/outputs/motions/
done
ls -la src/assets/motions/g1/ | grep the_feels_full

## 6. 追加学習（`model_23998` から 3,000 反復）

ログの先頭の `[train_sim]` 行に `motion_global_root_pos std 0.2` と、`Loading model checkpoint from: …model_23998.pt` が出ていることを確認してください。
学習を始めたら、すぐ下の監視セルを実行したままにします。

In [ ]:
%%bash
set -e
cd /content/unitree_rl_mjlab
if [ -f /content/train.pid ] && [ -d /proc/$(cat /content/train.pid) ]; then
  echo "[stop] 学習がすでに動いています（PID $(cat /content/train.pid)）。"; exit 1
fi
[ -f logs/rsl_rl/g1_tracking/$PREV_RUN/$PREV_CKPT ] || { echo "[stop] 読み込むチェックポイントがありません: $PREV_RUN/$PREV_CKPT"; exit 1; }
mkdir -p $P/logs
setsid nohup /content/venv_mjlab/bin/python $P/scripts/train_sim.py $TASK \
  --motion-file src/assets/motions/g1/$NAME.npz \
  --env.scene.num-envs 4096 \
  --agent.max-iterations $ITERS \
  --agent.logger tensorboard \
  --agent.run-name ${NAME}_ft2 \
  --agent.resume True --agent.load-run "$PREV_RUN" --agent.load-checkpoint "$PREV_CKPT" \
  --push-scale 0.5 \
  --action-rate-weight $ACTION_RATE --body-pos-std $BODY_POS_STD --body-ori-std $BODY_ORI_STD \
  --root-pos-std $ROOT_POS_STD \
  > $P/logs/train_$NAME.log 2>&1 &
echo $! > /content/train.pid
echo $ITERS > /content/train.iters
sleep 90; grep -E "train_sim|Loading model|Error|error" $P/logs/train_$NAME.log | head
echo "学習を開始しました（PID $(cat /content/train.pid)）。ログ: $P/logs/train_$NAME.log"

In [ ]:
# 監視セル：学習が終わるまで実行したままにする（止めても学習は止まらない）
import re, subprocess, time
from IPython.display import clear_output
pid = int(open('/content/train.pid').read())
log = f"{P}/logs/train_{NAME}.log"
alive = lambda: os.path.exists(f"/proc/{pid}")
sync = lambda: subprocess.run(['bash', f'{P}/scripts/sync_logs.sh', f'{L}/logs/rsl_rl', f'{P}/logs/rsl_rl'])
last = time.time()
while alive():
    time.sleep(60)
    txt = open(log, errors='ignore').read().splitlines()
    keep = [l.strip() for l in txt if re.search(r'Learning iteration|Mean episode length|Mean reward|ETA|Error|error', l)]
    clear_output(wait=True)
    print(time.strftime('%H:%M:%S'), f'学習中 (PID {pid})'); print('\n'.join(keep[-5:]))
    if time.time() - last > 600:
        sync(); last = time.time(); print('[sync] ドライブへ同期しました')
sync()
txt = open(log, errors='ignore').read()
print('学習プロセスが終了しました。ログの最後:'); print('\n'.join(txt.splitlines()[-15:]))

# 学習が正常に終わったかを確認（失敗していたら、ここで止めて後ろのセルを実行させない）
import glob
runs = sorted(glob.glob(f"{L}/logs/rsl_rl/g1_tracking/*_{NAME}*"), key=os.path.getmtime)
ckpts = glob.glob(f"{runs[-1]}/model_*.pt") if runs else []
its = [int(m) for m in re.findall(r'Learning iteration\s+(\d+)\s*/', txt)]
want = int(open('/content/train.iters').read()) if os.path.exists('/content/train.iters') else None
print(f"[check] run: {runs[-1] if runs else 'なし'} / チェックポイント {len(ckpts)} 個 / 最後の反復 {max(its) if its else 'なし'} / 予定 {want}")
if 'Traceback' in txt or not ckpts:
    raise RuntimeError('学習が途中でエラー終了しました。上のログの最後を確認してください（後ろのセルは実行しません）。')
if want and its and max(its) < want * 0.95:
    raise RuntimeError(f'学習が予定の反復数まで進まずに終わりました（{max(its)}/{want}）。ログを確認してください。')
print('[ok] 学習は正常に終わりました。評価と動画の書き出しに進みます。')

### 切断されたとき

1・2・5 を実行し直し、次のセルの `REMAIN` に残りの反復回数を入れて実行してから、監視セルをもう一度実行します。

In [ ]:
%%bash
REMAIN=0      # ← 残りの反復回数を入れたときだけ実行される
if [ "$REMAIN" = "0" ]; then echo "[skip] 再開しません（REMAIN=0）"; exit 0; fi
cd /content/unitree_rl_mjlab
RUN=$(basename $(ls -td logs/rsl_rl/g1_tracking/*_${NAME}_ft2* | head -1))
echo "再開する run: $RUN / $(ls -v logs/rsl_rl/g1_tracking/$RUN/model_*.pt | tail -1)"
setsid nohup /content/venv_mjlab/bin/python $P/scripts/train_sim.py $TASK \
  --motion-file src/assets/motions/g1/$NAME.npz --env.scene.num-envs 4096 \
  --agent.max-iterations $REMAIN --agent.logger tensorboard --agent.run-name ${NAME}_ft2_resume \
  --agent.resume True --agent.load-run "$RUN" --push-scale 0.5 \
  --action-rate-weight $ACTION_RATE --body-pos-std $BODY_POS_STD --body-ori-std $BODY_ORI_STD --root-pos-std $ROOT_POS_STD \
  > $P/logs/train_${NAME}_resume.log 2>&1 &
echo $! > /content/train.pid; echo $REMAIN > /content/train.iters; echo started

## 7. 評価

まずノートブック2・3と同じ評価（摩擦 0.8 は動画つき、0.5・1.0 は評価値）。続いて比較表を出します。
- **同じお手本（`the_feels_full_g25s`）の上で**、ノートブック3の方策（`model_23998`）と今回の方策を比べる（公平な比較）
- しゃがみ区間（9〜14 秒）だけの誤差も別に集計
- 参考に、ノートブック3の方策を元のお手本（`g25`）で評価した値も並べる

In [ ]:
%%bash
set -eo pipefail
cd /content/unitree_rl_mjlab
RUN=$(ls -td logs/rsl_rl/g1_tracking/*_${NAME}* | sed -n 1p)   # 一番新しい run（再開した run も含む）
[ -n "$RUN" ] || { echo "[stop] 学習の run が見つかりません"; exit 1; }
CKPT=$(ls -v $RUN/model_*.pt | tail -1)
TAG=$(basename $CKPT .pt)
[ -n "$CKPT" ] || { echo "[stop] チェックポイントが見つかりません（学習が終わっていない可能性）"; exit 1; }
echo "評価するチェックポイント: $CKPT"
mkdir -p $P/outputs/videos $P/outputs/eval
PY=/content/venv_mjlab/bin/python
$PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$NAME.npz \
  --checkpoint $CKPT --friction 0.8 --out $P/outputs/videos/policy_${NAME}_${TAG}.mp4 | tail -3
# 途中で転倒して動画が短い（または作られない）ときは、終了条件を外して全編の動画も書き出す（見せる用）
if python3 -c "import json,sys; sys.exit(0 if json.load(open('$P/outputs/videos/policy_${NAME}_${TAG}.json'))['success'] else 1)"; then
  echo "[ok] 転倒なしで最後まで踊れました"
else
  echo "[info] 転倒したため、終了条件を外した全編の動画も書き出します（_noterm.mp4）"
  $PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$NAME.npz \
    --checkpoint $CKPT --friction 0.8 --no-terminations --out $P/outputs/videos/policy_${NAME}_${TAG}_noterm.mp4 | grep -E "^\[" || true
fi
for F in 0.5 1.0; do
  $PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$NAME.npz \
    --checkpoint $CKPT --friction $F --no-video --out $P/outputs/eval/policy_${NAME}_${TAG}_mu$F.mp4 > /dev/null
done
python3 - <<EOF
import json, glob
fs = ["$P/outputs/videos/policy_${NAME}_${TAG}.json"] + sorted(glob.glob("$P/outputs/eval/policy_${NAME}_${TAG}_mu*.json"))
print(f"{'摩擦':>5} {'成功':>5} {'転倒時刻[s]':>10} {'関節誤差[rad]':>12} {'位置誤差[m]':>10}  止まった条件")
for f in fs:
    j = json.load(open(f))
    print(f"{j['friction']:>5} {str(j['success']):>5} {str(j['first_fall_s']):>10} {j['mean_joint_err_rad']:>12} {j['mean_body_pos_err_rel_m']:>10}  {j.get('fall_reason') or ''}")
import os
nt = "$P/outputs/videos/policy_${NAME}_${TAG}_noterm.json"
if os.path.exists(nt):
    j = json.load(open(nt))
    print(f"[終了条件なし・全編] 長さ {j['duration_s']} 秒 / 関節誤差 {j['mean_joint_err_rad']} rad / 位置誤差 平均 {j['mean_body_pos_err_rel_m']} m・最大 {j['max_body_pos_err_rel_m']} m")
EOF

In [ ]:
%%bash
set -e
cd /content/unitree_rl_mjlab
PY=/content/venv_mjlab/bin/python
NEW=$(ls -td logs/rsl_rl/g1_tracking/*_${NAME}_ft2* | sed -n 1p); NEWCK=$(ls -v $NEW/model_*.pt | tail -1)
PREVCK=logs/rsl_rl/g1_tracking/$PREV_RUN/$PREV_CKPT
mkdir -p $P/outputs/eval
run() {  # $1=チェックポイント $2=お手本 $3=出力名
  $PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$2.npz \
    --checkpoint $1 --friction 0.8 --no-video --out $P/outputs/eval/$3.mp4 > /dev/null
}
run $PREVCK $BASE_REF cmp4_prev_policy_g25
run $PREVCK $NAME     cmp4_prev_policy_g25s
run $NEWCK  $NAME     cmp4_new_policy_g25s
$PY - <<EOF
import json, numpy as np
rows = [("③ NB3方策 × g25（参考）", "cmp4_prev_policy_g25"),
        ("③' NB3方策 × g25s（お手本の補正のみ）", "cmp4_prev_policy_g25s"),
        ("④ 今回の方策 × g25s（補正＋追加学習）", "cmp4_new_policy_g25s")]
print(f"{'条件':<30} {'完走':>4} {'関節':>7} {'位置':>7} {'最大':>7} {'最大ドリフト':>10} {'act変化':>8} | {'9-14秒 関節':>11} {'位置':>7}")
for lab, n in rows:
    j = json.load(open(f"$P/outputs/eval/{n}.json"))
    a = np.loadtxt(f"$P/outputs/eval/{n}.csv", delimiter=",", skiprows=1)
    m = (a[:, 0] >= 9) & (a[:, 0] < 14)
    print(f"{lab:<30} {str(j['success']):>4} {j['mean_joint_err_rad']:>7} {j['mean_body_pos_err_rel_m']:>7} {j['max_body_pos_err_rel_m']:>7}"
          f" {j['max_root_xy_drift_m']:>10} {j['mean_action_rate']:>8} | {a[m,1].mean():>11.4f} {a[m,2].mean():>7.4f}")
EOF

In [ ]:
import glob
vids = sorted(glob.glob(f"{P}/outputs/videos/policy_{NAME}_model_*.mp4"), key=os.path.getmtime)
if not vids:
    print("動画がありません（評価で開始直後に止まった可能性）。3-6 の出力を確認してください。")
else:
    latest = vids[-1]
    print(latest, f"({os.path.getsize(latest) / 1e6:.1f} MB)")
    display(show(latest))

## 8. 3画面すべて正面の比較動画

左：人のメッシュ（**ノートブック 04b** で作った `2_global_front.mp4`）／中央：お手本（正面プレビュー）／右：今回の方策（正面）。
出力は `comparison_the_feels_full_g25s_front_all.mp4` です。04b がまだなら、このセルは止まって知らせます。

In [ ]:
%%bash
set -e
cd /content/unitree_rl_mjlab
PY=/content/venv_mjlab/bin/python
HUMAN=$P/outputs/gvhmr/the_feels_full/2_global_front.mp4
REF=$P/outputs/motions/${NAME}_retarget_preview_front.mp4
[ -f $HUMAN ] || { echo "[stop] $HUMAN がありません。先にノートブック 04b を実行してください"; exit 1; }
[ -f $REF ] || { echo "[stop] $REF がありません。手順 3 を実行してください"; exit 1; }
# 右：今回の方策を正面から（評価値も同時に出る）
NEW=$(ls -td logs/rsl_rl/g1_tracking/*_${NAME}_ft2* | sed -n 1p); NEWCK=$(ls -v $NEW/model_*.pt | tail -1)
ROBOT=$P/outputs/videos/front_policy_${NAME}_$(basename $NEWCK .pt).mp4
$PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$NAME.npz \
  --checkpoint $NEWCK --friction 0.8 --azimuth $AZ --elevation $EL --distance $DIST --out $ROBOT | grep -E '"success"|"mean_body_pos_err_rel_m"' || true
# 3 画面を結合
START=0 bash $P/scripts/make_comparison.sh $HUMAN $REF $ROBOT $P/input/the_feels_full.mp4 \
  $P/outputs/videos/comparison_${NAME}_front_all.mp4
ls -la $P/outputs/videos/comparison_${NAME}_front_all.mp4

In [ ]:
show(f"{P}/outputs/videos/comparison_{NAME}_front_all.mp4", width=1100)

## 付録：報酬の調整の効果を切り分ける比較実験（任意・約 1.8 時間）

ノートブック3の改善（関節誤差 −37%）が「報酬の調整」によるのか「反復を 4,000 増やしたこと」によるのかを分けるため、
**報酬を既定値のまま**、ノートブック3と同じ条件（`model_19999` から、お手本 `g25`、4,000 反復）で学習します。
本編（手順 6〜8）が終わってから実行してください（GPU を同時に使うと両方遅くなります）。

In [ ]:
%%bash
set -e
cd /content/unitree_rl_mjlab
if [ -f /content/train.pid ] && [ -d /proc/$(cat /content/train.pid) ]; then echo "[stop] 学習がすでに動いています"; exit 1; fi
setsid nohup /content/venv_mjlab/bin/python $P/scripts/train_sim.py $TASK \
  --motion-file src/assets/motions/g1/$BASE_REF.npz --env.scene.num-envs 4096 \
  --agent.max-iterations 4000 --agent.logger tensorboard --agent.run-name the_feels_full_g25_abl \
  --agent.resume True --agent.load-run 2026-10-17_08-14-52_the_feels_full --agent.load-checkpoint model_19999.pt \
  --push-scale 0.5 \
  > $P/logs/train_the_feels_full_g25_abl.log 2>&1 &
echo $! > /content/train.pid; echo 4000 > /content/train.iters
sleep 90; grep -E "train_sim|Loading model" $P/logs/train_the_feels_full_g25_abl.log | head

In [ ]:
ABL = 'the_feels_full_g25_abl'   # 切り分け実験の監視
# 監視セル：学習が終わるまで実行したままにする（止めても学習は止まらない）
import re, subprocess, time
from IPython.display import clear_output
pid = int(open('/content/train.pid').read())
log = f"{P}/logs/train_{ABL}.log"
alive = lambda: os.path.exists(f"/proc/{pid}")
sync = lambda: subprocess.run(['bash', f'{P}/scripts/sync_logs.sh', f'{L}/logs/rsl_rl', f'{P}/logs/rsl_rl'])
last = time.time()
while alive():
    time.sleep(60)
    txt = open(log, errors='ignore').read().splitlines()
    keep = [l.strip() for l in txt if re.search(r'Learning iteration|Mean episode length|Mean reward|ETA|Error|error', l)]
    clear_output(wait=True)
    print(time.strftime('%H:%M:%S'), f'学習中 (PID {pid})'); print('\n'.join(keep[-5:]))
    if time.time() - last > 600:
        sync(); last = time.time(); print('[sync] ドライブへ同期しました')
sync()
txt = open(log, errors='ignore').read()
print('学習プロセスが終了しました。ログの最後:'); print('\n'.join(txt.splitlines()[-15:]))

# 学習が正常に終わったかを確認（失敗していたら、ここで止めて後ろのセルを実行させない）
import glob
runs = sorted(glob.glob(f"{L}/logs/rsl_rl/g1_tracking/*_{ABL}*"), key=os.path.getmtime)
ckpts = glob.glob(f"{runs[-1]}/model_*.pt") if runs else []
its = [int(m) for m in re.findall(r'Learning iteration\s+(\d+)\s*/', txt)]
want = int(open('/content/train.iters').read()) if os.path.exists('/content/train.iters') else None
print(f"[check] run: {runs[-1] if runs else 'なし'} / チェックポイント {len(ckpts)} 個 / 最後の反復 {max(its) if its else 'なし'} / 予定 {want}")
if 'Traceback' in txt or not ckpts:
    raise RuntimeError('学習が途中でエラー終了しました。上のログの最後を確認してください（後ろのセルは実行しません）。')
if want and its and max(its) < want * 0.95:
    raise RuntimeError(f'学習が予定の反復数まで進まずに終わりました（{max(its)}/{want}）。ログを確認してください。')
print('[ok] 学習は正常に終わりました。評価と動画の書き出しに進みます。')

In [ ]:
%%bash
set -e
cd /content/unitree_rl_mjlab
PY=/content/venv_mjlab/bin/python
ABL=$(ls -td logs/rsl_rl/g1_tracking/*_the_feels_full_g25_abl* | sed -n 1p); CK=$(ls -v $ABL/model_*.pt | tail -1)
$PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$BASE_REF.npz \
  --checkpoint $CK --friction 0.8 --no-video --out $P/outputs/eval/cmp_abl_default_reward_new_ref.mp4 > /dev/null
$PY - <<EOF
import json
rows = [("② 旧方策（19999）× g25", "cmp_old_policy_new_ref"), ("既定の報酬で +4000 反復 × g25", "cmp_abl_default_reward_new_ref"),
        ("③ 調整した報酬で +4000 反復 × g25", "cmp_new_policy_new_ref")]
for lab, n in rows:
    j = json.load(open(f"$P/outputs/eval/{n}.json"))
    print(f"{lab:<34} 関節 {j['mean_joint_err_rad']:<7} 位置 {j['mean_body_pos_err_rel_m']:<7} 最大ドリフト {j['max_root_xy_drift_m']:<6} act変化 {j['mean_action_rate']}")
EOF